In [1]:
import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm

K = 5

dataset = pd.read_parquet("data/rag_qa_ptbr_chunk_embeddings.parquet")
embedded_questions_df = pd.read_parquet("data/encoded_questions.parquet")

# -----------------------------
# Documents
# -----------------------------

index_df = dataset[["document_id", "embedding"]].copy()
index_df = index_df[index_df["embedding"].notna()].reset_index(drop=True)

doc_ids = index_df["document_id"].astype(str).to_numpy()
X = np.vstack(index_df["embedding"].values).astype("float32")

faiss.normalize_L2(X)

n_docs, dim = X.shape

print("Document vectors:", X.shape)


# -----------------------------
# Queries
# -----------------------------

queries_df = embedded_questions_df[["document_id", "question_embedding"]].copy()
queries_df = queries_df[queries_df["question_embedding"].notna()].reset_index(drop=True)

gold_doc_ids = queries_df["document_id"].astype(str).to_numpy()
Q = np.vstack(queries_df["question_embedding"].values).astype("float32")

faiss.normalize_L2(Q)

assert Q.shape[1] == dim

print("Query vectors:", Q.shape)

Document vectors: (1420215, 1024)
Query vectors: (17138, 1024)


In [2]:
def evaluate_indices(indices, doc_ids, gold_doc_ids, k=5):
    hits = []
    reciprocal_ranks = []

    rows = []

    for q_idx in range(len(gold_doc_ids)):
        gold = gold_doc_ids[q_idx]

        retrieved_doc_ids = [
            doc_ids[idx]
            for idx in indices[q_idx]
            if idx != -1
        ]

        if gold in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(gold) + 1
            hit = 1
            rr = 1.0 / rank
        else:
            rank = None
            hit = 0
            rr = 0.0

        hits.append(hit)
        reciprocal_ranks.append(rr)

        rows.append({
            "query_index": q_idx,
            "gold_document_id": gold,
            "retrieved_document_ids": retrieved_doc_ids,
            "hit@5": hit,
            "rank@5": rank,
            "reciprocal_rank@5": rr,
        })

    eval_df = pd.DataFrame(rows)

    return {
        "hit_rate@5": float(np.mean(hits)),
        "mrr@5": float(np.mean(reciprocal_ranks)),
        "eval_df": eval_df,
    }


def mb(num_bytes):
    return num_bytes / 1024**2

In [3]:
if faiss.get_num_gpus() == 0:
    raise RuntimeError("FAISS does not see a GPU. You need faiss-gpu, not faiss-cpu.")

gpu_id = 0
res = faiss.StandardGpuResources()

index_fp32_cpu = faiss.IndexFlatIP(dim)
index_fp32 = faiss.index_cpu_to_gpu(res, gpu_id, index_fp32_cpu)

index_fp32.add(X)

scores_fp32, indices_fp32 = index_fp32.search(Q, K)

metrics_fp32 = evaluate_indices(indices_fp32, doc_ids, gold_doc_ids, k=K)

fp32_size_bytes = X.nbytes

print("FP32 Hit Rate@5:", metrics_fp32["hit_rate@5"])
print("FP32 MRR@5:", metrics_fp32["mrr@5"])
print("FP32 vector size MB:", mb(fp32_size_bytes))

FP32 Hit Rate@5: 0.8277511961722488
FP32 MRR@5: 0.7457414322947057
FP32 vector size MB: 5547.71484375


In [4]:
config = faiss.GpuIndexFlatConfig()
config.device = gpu_id
config.useFloat16 = True

index_fp16 = faiss.GpuIndexFlatIP(res, dim, config)
index_fp16.add(X)

scores_fp16, indices_fp16 = index_fp16.search(Q, K)

metrics_fp16 = evaluate_indices(indices_fp16, doc_ids, gold_doc_ids, k=K)

fp16_size_bytes = n_docs * dim * 2

print("FP16 Hit Rate@5:", metrics_fp16["hit_rate@5"])
print("FP16 MRR@5:", metrics_fp16["mrr@5"])
print("FP16 approximate vector size MB:", mb(fp16_size_bytes))

FP16 Hit Rate@5: 0.8277511961722488
FP16 MRR@5: 0.7457443497879954
FP16 approximate vector size MB: 2773.857421875


In [5]:
def choose_pq_m(dim):
    for m in [128, 96, 80, 64, 48, 40, 32, 24, 20, 16, 12, 8]:
        if dim % m == 0:
            return m
    raise ValueError(f"Could not find a clean PQ M divisor for dim={dim}")


M = choose_pq_m(dim)
NBITS = 8

# Conservative nlist for small or medium datasets
NLIST = min(4096, max(16, int(np.sqrt(n_docs))))

print("PQ M:", M)
print("PQ bits per subvector:", NBITS)
print("IVF nlist:", NLIST)

quantizer = faiss.IndexFlatIP(dim)

index_ivfpq_cpu = faiss.IndexIVFPQ(
    quantizer,
    dim,
    NLIST,
    M,
    NBITS,
    faiss.METRIC_INNER_PRODUCT,
)

index_ivfpq_cpu.train(X)

try:
    index_ivfpq = faiss.index_cpu_to_gpu(res, gpu_id, index_ivfpq_cpu)
    print("Using IVFPQ on GPU")
except Exception as e:
    print("Could not move IVFPQ to GPU, using CPU:", repr(e))
    index_ivfpq = index_ivfpq_cpu

index_ivfpq.add(X)

# Higher nprobe usually improves quality but increases latency
index_ivfpq.nprobe = min(16, NLIST)

scores_ivfpq, indices_ivfpq = index_ivfpq.search(Q, K)

metrics_ivfpq = evaluate_indices(indices_ivfpq, doc_ids, gold_doc_ids, k=K)

# Approximate vector-code storage only, excluding IVF ids/centroids/overhead
pq_code_size_bytes = n_docs * M * NBITS // 8

print("IVFPQ Hit Rate@5:", metrics_ivfpq["hit_rate@5"])
print("IVFPQ MRR@5:", metrics_ivfpq["mrr@5"])
print("IVFPQ approximate code size MB:", mb(pq_code_size_bytes))

PQ M: 128
PQ bits per subvector: 8
IVF nlist: 1191
Could not move IVFPQ to GPU, using CPU: RuntimeError("Error in void faiss::gpu::GpuIndexIVFPQ::verifyPQSettings_() const at /project/faiss/faiss/gpu/GpuIndexIVFPQ.cu:580: Error: 'ivfpqConfig_.interleavedLayout || IVFPQ::isSupportedPQCodeLength(subQuantizers_)' failed: Number of bytes per encoded vector / sub-quantizers (128) is not supported")
IVFPQ Hit Rate@5: 0.7298401213677208
IVFPQ MRR@5: 0.6524857042828801
IVFPQ approximate code size MB: 173.3660888671875


In [6]:
# index_sq8_cpu = faiss.IndexScalarQuantizer(
#     dim,
#     faiss.ScalarQuantizer.QT_8bit,
#     faiss.METRIC_INNER_PRODUCT,
# )
#
# index_sq8_cpu.train(X)
#
# try:
#     index_sq8 = faiss.index_cpu_to_gpu(res, gpu_id, index_sq8_cpu)
#     print("Using SQ8 on GPU")
# except Exception as e:
#     print("Could not move SQ8 to GPU, using CPU:", repr(e))
#     index_sq8 = index_sq8_cpu
#
# index_sq8.add(X)
#
# scores_sq8, indices_sq8 = index_sq8.search(Q, K)
#
# metrics_sq8 = evaluate_indices(indices_sq8, doc_ids, gold_doc_ids, k=K)
#
# sq8_size_bytes = n_docs * dim * 1
#
# print("SQ8 Hit Rate@5:", metrics_sq8["hit_rate@5"])
# print("SQ8 MRR@5:", metrics_sq8["mrr@5"])
# print("SQ8 approximate vector size MB:", mb(sq8_size_bytes))

In [8]:
from turbovec import TurboQuantIndex
import numpy as np
import pandas as pd

BIT_WIDTHS = [4, 3, 2]

turbovec_results = []

for bit_width in BIT_WIDTHS:
    print(f"\nRunning TurboVec with {bit_width}-bit compression...")

    tq_index = TurboQuantIndex(dim=dim, bit_width=bit_width)

    # TurboVec expects float32 vectors
    tq_index.add(X.astype("float32"))

    scores_tq, indices_tq = tq_index.search(
        Q.astype("float32"),
        k=K
    )

    metrics_tq = evaluate_indices(
        indices_tq,
        doc_ids,
        gold_doc_ids,
        k=K
    )

    tq_size_bytes = n_docs * dim * bit_width // 8

    row = {
        "index": f"turbovec_{bit_width}bit",
        "approx_vector_storage_mb": mb(tq_size_bytes),
        "compression_vs_fp32": (n_docs * dim * 4) / tq_size_bytes,
        "hit_rate@5": metrics_tq["hit_rate@5"],
        "mrr@5": metrics_tq["mrr@5"],
    }

    turbovec_results.append(row)

    print(f"TurboVec {bit_width}-bit Hit Rate@5: {row['hit_rate@5']:.4f}")
    print(f"TurboVec {bit_width}-bit MRR@5:      {row['mrr@5']:.4f}")
    print(f"TurboVec {bit_width}-bit size MB:    {row['approx_vector_storage_mb']:.2f}")
    print(f"Compression vs fp32:                 {row['compression_vs_fp32']:.2f}x")

turbovec_summary_df = pd.DataFrame(turbovec_results)




Running TurboVec with 4-bit compression...
TurboVec 4-bit Hit Rate@5: 0.8275
TurboVec 4-bit MRR@5:      0.7463
TurboVec 4-bit size MB:    693.46
Compression vs fp32:                 8.00x

Running TurboVec with 3-bit compression...
TurboVec 3-bit Hit Rate@5: 0.8274
TurboVec 3-bit MRR@5:      0.7449
TurboVec 3-bit size MB:    520.10
Compression vs fp32:                 10.67x

Running TurboVec with 2-bit compression...
TurboVec 2-bit Hit Rate@5: 0.8258
TurboVec 2-bit MRR@5:      0.7432
TurboVec 2-bit size MB:    346.73
Compression vs fp32:                 16.00x


,index,approx_vector_size_mb,compression_vs_fp32,hit_rate@5,mrr@5
0,turbovec_4bit,693.464355,8.000000,0.827459,0.746257
1,turbovec_3bit,520.098267,10.666667,0.827401,0.744934
2,turbovec_2bit,346.732178,16.000000,0.825826,0.743224


In [24]:
summary_df = pd.DataFrame([
    {
        "index": "faiss_gpu_flat_fp32",
        "approx_vector_storage_mb": mb(fp32_size_bytes),
        "compression_vs_fp32": fp32_size_bytes / fp32_size_bytes,
        "hit_rate@5": metrics_fp32["hit_rate@5"],
        "mrr@5": metrics_fp32["mrr@5"],
    },
    {
        "index": "faiss_gpu_flat_fp16",
        "approx_vector_storage_mb": mb(fp16_size_bytes),
        "compression_vs_fp32": fp32_size_bytes / fp16_size_bytes,
        "hit_rate@5": metrics_fp16["hit_rate@5"],
        "mrr@5": metrics_fp16["mrr@5"],
    },
    {
        "index": f"faiss_ivfpq_M{M}_nbits{NBITS}",
        "approx_vector_storage_mb": mb(pq_code_size_bytes),
        "compression_vs_fp32": fp32_size_bytes / pq_code_size_bytes,
        "hit_rate@5": metrics_ivfpq["hit_rate@5"],
        "mrr@5": metrics_ivfpq["mrr@5"],
    },

])
summary_df = pd.concat(
    [summary_df, turbovec_summary_df],
    ignore_index=True
)
summary_df.to_csv("data/compression_results.csv", index=False)